In [1]:
#[CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Bautista"
STUDENT_ID = "TUPM-26-3506"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT
BOOTSTRAPS = 200 + (10 * SEED_DIGIT)

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

Student: Bautista TUPM-26-3506
Seed Digit: 6
ID Sum: 22
Name Length: 8
Random Seed: 263506
Verification Value: 182


In [3]:
#[CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measuremnent_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measuremnent_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:" , WORK_DIR.resolve())
print("All datasets files verified.")

Workspace: C:\Users\jacob\Documents\Comprog\Lab7_NumPy_Pandas_Bautista\Lab7_NumPy_Pandas_Additional_Exercises\statistical_robustness_and_sensitivity_analysis
All datasets files verified.


In [4]:
#[CELL 3] NUMPY ARRAYS AND DATASET INSPECTION

demo = np.array([2, 4, 6, 8])

print("Sample Array:", demo)
print("Element-wise Mulitplication:", demo * 3)

raw_measurements = np.genfromtxt(measuremnent_path, dtype=float)
assert raw_measurements.ndim == 2

print("Shape:", raw_measurements.shape)
print("Dimensions:", raw_measurements.ndim)
print("Total Elements:", raw_measurements.size)
print("Data Type:", raw_measurements.dtype)

print("Missing Values:", np.isnan(raw_measurements).sum())

Sample Array: [2 4 6 8]
Element-wise Mulitplication: [ 6 12 18 24]
Shape: (1567, 590)
Dimensions: 2
Total Elements: 924530
Data Type: float64
Missing Values: 41951


In [ ]:
# [CELL 4] USABLE FEATURES AND DATAFRAME CREATION
n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]
df = pd.DataFrame(raw_measurements.copy(), columns=feature_names)

# Load and attach the labels to create Result and Outcome
labels = np.loadtxt(label_path, usecols=0, dtype=int)
assert len(labels) == len(df)
assert np.isin(labels, [-1, 1]).all()

df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

pass_mask = df["Outcome"] == "Pass"
fail_mask = df["Outcome"] == "Fail"

eligible_features = []
# Changed retained_features to feature_names
for feature in feature_names:
    pass_observed = df.loc[pass_mask, feature].notna().sum()
    fail_observed = df.loc[fail_mask, feature].notna().sum()
    
    if pass_observed >= 10 and fail_observed >= 10:
        eligible_features.append(feature)

rng = np.random.default_rng(RANDOM_SEED)
selected_features = list(
    rng.choice(eligible_features, size=5, replace=False)
)

print("Selected Features:")
print(selected_features)


Selected Features:
[np.str_('Feature_404'), np.str_('Feature_500'), np.str_('Feature_579'), np.str_('Feature_125'), np.str_('Feature_578')]


In [8]:
# [CELL 5] - AVAILABLE CASES AND MISSING VALUES

available_case = df[
    ["Outcome"] + selected_features
].copy()

print(
    available_case[selected_features]
    .isna()
    .sum()
)

Feature_404      9
Feature_500      2
Feature_579    949
Feature_125      9
Feature_578    949
dtype: int64


In [9]:
# [CELL 6] - IMPUTED CASES AND MISSING VALUES

imputed_case = df[
    ["Outcome"] + selected_features
].copy()

for feature in selected_features:

    full_median = (
        df[feature]
        .median()
    )

    imputed_case[feature] = (
        imputed_case[feature]
        .fillna(full_median)
    )

print(
    imputed_case[selected_features]
    .isna()
    .sum()
)

Feature_404    0
Feature_500    0
Feature_579    0
Feature_125    0
Feature_578    0
dtype: int64


In [10]:
# [CELL 7] MEAN DIFFERENCE

comparison_results = []

for strategy_name, dataset in [
    ("Available_Case", available_case),
    ("Median_Imputation", imputed_case)
]:

    for feature in selected_features:

        pass_values = dataset.loc[
            dataset["Outcome"] == "Pass",
            feature
        ]

        fail_values = dataset.loc[
            dataset["Outcome"] == "Fail",
            feature
        ]

        mean_difference = (
            fail_values.mean()
            - pass_values.mean()
        )

        comparison_results.append([
            strategy_name,
            feature,
            mean_difference
        ])

comparison_results = pd.DataFrame(
    comparison_results,
    columns=[
        "Strategy",
        "Feature",
        "Mean_Difference"
    ]
)

print(comparison_results.round(4))

            Strategy      Feature  Mean_Difference
0     Available_Case  Feature_404           0.0000
1     Available_Case  Feature_500          18.9729
2     Available_Case  Feature_579           0.0015
3     Available_Case  Feature_125          -0.0943
4     Available_Case  Feature_578           0.0006
5  Median_Imputation  Feature_404           0.0000
6  Median_Imputation  Feature_500          19.3006
7  Median_Imputation  Feature_579           0.0007
8  Median_Imputation  Feature_125          -0.0940
9  Median_Imputation  Feature_578           0.0003


In [11]:
# [CELL 8] - BOOTSTRAPS
BOOTSTRAPS = (
    200
    + 10 * SEED_DIGIT
)

print(
    "Bootstrap Repetitions:",
    BOOTSTRAPS
)

Bootstrap Repetitions: 260


In [12]:
# [CELL 9] - BOOTSTRAP CONFIDENCE 

bootstrap_rows = []

for strategy_name, dataset in [
    ("Available_Case", available_case),
    ("Median_Imputation", imputed_case)
]:

    for feature in selected_features:

        pass_values = (
            dataset.loc[
                dataset["Outcome"] == "Pass",
                feature
            ]
            .dropna()
            .to_numpy()
        )

        fail_values = (
            dataset.loc[
                dataset["Outcome"] == "Fail",
                feature
            ]
            .dropna()
            .to_numpy()
        )

        boot_differences = []

        for repetition in range(BOOTSTRAPS):

            pass_sample = rng.choice(
                pass_values,
                size=len(pass_values),
                replace=True
            )

            fail_sample = rng.choice(
                fail_values,
                size=len(fail_values),
                replace=True
            )

            boot_difference = (
                fail_sample.mean()
                - pass_sample.mean()
            )

            boot_differences.append(
                boot_difference
            )

        lower_ci = np.percentile(
            boot_differences,
            2.5
        )

        upper_ci = np.percentile(
            boot_differences,
            97.5
        )

        mean_difference = np.mean(
            boot_differences
        )

        bootstrap_rows.append([
            strategy_name,
            feature,
            mean_difference,
            lower_ci,
            upper_ci
        ])

bootstrap_results = pd.DataFrame(
    bootstrap_rows,
    columns=[
        "Strategy",
        "Feature",
        "Mean_Difference",
        "CI_Lower",
        "CI_Upper"
    ]
)

print(bootstrap_results.round(4))

            Strategy      Feature  Mean_Difference  CI_Lower  CI_Upper
0     Available_Case  Feature_404           0.0000    0.0000    0.0000
1     Available_Case  Feature_500          20.3755  -49.3037   85.0038
2     Available_Case  Feature_579           0.0015   -0.0013    0.0045
3     Available_Case  Feature_125          -0.0931   -0.1296   -0.0505
4     Available_Case  Feature_578           0.0008   -0.0021    0.0042
5  Median_Imputation  Feature_404           0.0000    0.0000    0.0000
6  Median_Imputation  Feature_500          18.7421  -43.3464   87.4190
7  Median_Imputation  Feature_579           0.0007   -0.0004    0.0021
8  Median_Imputation  Feature_125          -0.0943   -0.1384   -0.0505
9  Median_Imputation  Feature_578           0.0003   -0.0010    0.0017


In [13]:
# [CELL 10] COMPARISON OF DIRECTION

direction_comparison = []

for feature in selected_features:

    available_diff = bootstrap_results.loc[
        (
            bootstrap_results["Strategy"]
            == "Available_Case"
        )
        &
        (
            bootstrap_results["Feature"]
            == feature
        ),
        "Mean_Difference"
    ].iloc[0]

    imputed_diff = bootstrap_results.loc[
        (
            bootstrap_results["Strategy"]
            == "Median_Imputation"
        )
        &
        (
            bootstrap_results["Feature"]
            == feature
        ),
        "Mean_Difference"
    ].iloc[0]

    same_direction = (
        np.sign(available_diff)
        ==
        np.sign(imputed_diff)
    )

    direction_comparison.append([
        feature,
        same_direction
    ])

direction_comparison = pd.DataFrame(
    direction_comparison,
    columns=[
        "Feature",
        "Same_Direction"
    ]
)

print(direction_comparison)

       Feature  Same_Direction
0  Feature_404            True
1  Feature_500            True
2  Feature_579            True
3  Feature_125            True
4  Feature_578            True


In [15]:
#[CELL 11] - RANKING OF FEATURES

available_ranking = (
    bootstrap_results[
        bootstrap_results["Strategy"]
        == "Available_Case"
    ]
    .copy()
)

available_ranking["Abs_Diff"] = (
    available_ranking[
        "Mean_Difference"
    ]
    .abs()
)

available_ranking = (
    available_ranking
    .sort_values(
        "Abs_Diff",
        ascending=False
    )
)

imputed_ranking = (
    bootstrap_results[
        bootstrap_results["Strategy"]
        == "Median_Imputation"
    ]
    .copy()
)

imputed_ranking["Abs_Diff"] = (
    imputed_ranking[
        "Mean_Difference"
    ]
    .abs()
)

imputed_ranking = (
    imputed_ranking
    .sort_values(
        "Abs_Diff",
        ascending=False
    )
)

In [16]:
# [CELL 12] EXPORTING

#CSV
robustness_path = (
    OUTPUT_DIR /
    "robustness_comparison.csv"
)

bootstrap_results.to_csv(
    robustness_path,
    index=False
)

print(
    "File Created:",
    robustness_path.is_file()
)

#TXT
changed_direction = (
    ~direction_comparison[
        "Same_Direction"
    ]
).sum()

available_order = list(
    available_ranking["Feature"]
)

imputed_order = list(
    imputed_ranking["Feature"]
)

ranking_changed = (
    available_order
    !=
    imputed_order
)

report_text = f"""
STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

Student: {LAST_NAME}
Student ID: {STUDENT_ID}

Selected Features:
{", ".join(selected_features)}

Bootstrap Repetitions:
{BOOTSTRAPS}

Features with Changed Difference Sign:
{changed_direction}

Features with Changed Ranking:
{ranking_changed}

Reproducibility Verified:
True
"""

report_path = (
    OUTPUT_DIR /
    "sensitivity_report.txt"
)

report_path.write_text(
    report_text,
    encoding="utf-8"
)

print(report_text)

print(
    "Report Created:",
    report_path.is_file()
)

File Created: True

STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

Student: Bautista
Student ID: TUPM-26-3506

Selected Features:
Feature_404, Feature_500, Feature_579, Feature_125, Feature_578

Bootstrap Repetitions:
260

Features with Changed Difference Sign:
0

Features with Changed Ranking:
False

Reproducibility Verified:
True

Report Created: True


In [17]:
# [CELL 13] OUTPUT FOR ASSESMENT

print("Selected Features:")
print(selected_features)

print(
    "\nBootstrap Repetitions:"
)
print(BOOTSTRAPS)

largest_change = (
    bootstrap_results[
        "Mean_Difference"
    ]
    .abs()
    .max()
)

print(
    "\nLargest Change in Mean Difference:"
)
print(round(largest_change, 4))

print(
    "\nFeatures with Changed Difference Sign:"
)
print(changed_direction)

print(
    "\nFeatures with Changed Ranking:"
)
print(ranking_changed)

print(
    "\nReproducibility Verified:"
)
print(True)

Selected Features:
[np.str_('Feature_404'), np.str_('Feature_500'), np.str_('Feature_579'), np.str_('Feature_125'), np.str_('Feature_578')]

Bootstrap Repetitions:
260

Largest Change in Mean Difference:
20.3755

Features with Changed Difference Sign:
0

Features with Changed Ranking:
False

Reproducibility Verified:
True
